In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 1. Setup label list

In [3]:
import os, cv2, json, time
import numpy as np
import torch
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
!pip install ultralytics
from ultralytics import YOLO
import pandas as pd
import timm

LABELS = [
    "A","B","C","D","E","F","G","H","I","J","K",
    "L","M","N","O","P","Q","R","S","T","U","V",
    "W","X","Y","Z","delete","nothing","space"
]
IMG_SIZE = 224

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 21.6 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


# 2. Hàm load model linh hoạt (TF hoặc Torch)

In [4]:
!pip install transformers


In [5]:
!pip install timm

In [6]:
def load_vit_timm(model_path):
    import torch
    import timm

    print("🟣 Loading ViT (timm) state_dict... fixing DDP prefixes...")

    # 1. Tạo mô hình đúng kiến trúc
    model = timm.create_model(
        'vit_base_patch16_224',
        pretrained=False,
        num_classes=29
    )

    # 2. Load raw checkpoint
    state = torch.load(model_path, map_location="cpu")

    # nếu checkpoint dạng {"state_dict": ...}
    if "state_dict" in state:
        state = state["state_dict"]

    # 3. Remove prefix "module." nếu có
    new_state = {}
    for k, v in state.items():
        new_k = k.replace("module.", "")  # remove prefix
        new_state[new_k] = v

    # 4. Load lại state_dict đã clean
    model.load_state_dict(new_state, strict=True)

    model.eval()
    return model


In [7]:
from ultralytics import YOLO

def load_model_flexible(model_path, model_type):

    # TensorFlow model
    if model_type == "tf":
        print("🔵 Loading TensorFlow/Keras model...")
        return tf.keras.models.load_model(model_path, compile=False)

    # YOLO (detector/classifier)
    if model_type == "yolo":
        print("🟠 Loading YOLO model (.pt)...")
        from ultralytics import YOLO
        return YOLO(model_path)

    # PyTorch ViT (.pth)
    elif model_type == "vit":
        return load_vit_timm(model_path)

    raise ValueError(f"Unknown model_type: {model_type}")



# 3. Preprocess theo từng loại model

In [8]:
def preprocess_tf(img):
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img.astype("float32") / 255.0
    return np.expand_dims(img, axis=0)

def preprocess_torch(img):
    # chỉ resize, không chuyển dạng, để YOLO tự xử lý normalization
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    return img   # numpy HWC

def preprocess_vit(img):
    # Resize và BGR->RGB
    img = cv2.resize(img, (224,224))
    img = img[:, :, ::-1].copy()  # sang RGB

    # float32
    img = img.astype("float32") / 255.0

    # ImageNet normalization
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    img = (img - mean) / std

    # HWC -> CHW
    img = np.transpose(img, (2,0,1))

    # tensor float32
    img = torch.tensor(img, dtype=torch.float32).unsqueeze(0)
    return img




# 4. Hàm dự đoán

In [9]:
def predict_single(model, img, model_type):
    # -------------------------
    # 1. TensorFlow (Keras)
    # -------------------------
    if model_type == "tf":
        # Đảm bảo img là RGB
        if img.ndim == 2:  # grayscale
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

        if img.shape[-1] == 1:  # (224,224,1)
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

        # Resize nếu cần
        if img.shape[:2] != (224, 224):
            img = cv2.resize(img, (224, 224))

        # BGR → RGB
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Chuẩn hóa
        img = img.astype("float32") / 255.0

        # Thêm batch dimension
        if img.ndim == 3:
            img = np.expand_dims(img, axis=0)  # (1,224,224,3)

        pred = model.predict(img, verbose=0)[0]
        return pred.tolist()

    # -------------------------
    # 2. YOLO (classification / detection)
    # -------------------------
    if model_type == "yolo":
        results = model(img, verbose=False)[0]

        # YOLO classification
        if results.probs is not None:
            return results.probs.data.cpu().numpy().tolist()

        # YOLO detection
        boxes = results.boxes
        if boxes is None or len(boxes) == 0:
            return None

        cls_ids = boxes.cls.cpu().numpy().astype(int)
        confs   = boxes.conf.cpu().numpy()
        if len(confs) == 0:
            return None

        best = np.argmax(confs)
        return cls_ids[best]

    # -------------------------
    # 3. Vision Transformer
    # -------------------------
    if model_type == "vit":
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (224,224))
        img = img.astype("float32") / 255.0
        img = np.expand_dims(img, axis=0)
        img = torch.tensor(img).permute(0,3,1,2).float().to(next(model.parameters()).device)

        with torch.no_grad():
            logits = model(img)
            probs = torch.softmax(logits[0], dim=0).cpu().numpy()
        return probs.tolist()


# 5. Evaluate một video (với YOLO hand crop)

In [10]:
def evaluate_single_video(video_path, true_label, model, model_type, hand_detector, save_dir):

    cap = cv2.VideoCapture(video_path)
    pred_list = []
    latencies = []
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_idx += 1

        # 1. Detect hand
        det_hand = hand_detector(frame)[0]
        if len(det_hand.boxes) == 0:
            continue

        boxes = det_hand.boxes.xyxy.cpu().numpy()
        areas = (boxes[:, 2]-boxes[:, 0]) * (boxes[:, 3]-boxes[:, 1])
        x1, y1, x2, y2 = boxes[np.argmax(areas)].astype(int)

        roi = frame[y1:y2, x1:x2]
        if roi.size == 0:
            continue

        # 2. Resize ROI cho TF/VIT (YOLO tự xử lý)
        if model_type in ["tf", "vit"]:
            roi_input = cv2.resize(roi, (224, 224))
        else:
            roi_input = roi

        # 3. Predict
        t0 = time.time()
        out = predict_single(model, roi_input, model_type)
        t1 = time.time()
        latencies.append((t1 - t0) * 1000)

        if out is None:
            continue

        # 4. Convert output → class label
        if model_type == "yolo":
            if isinstance(out, list):
                pred_idx = int(np.argmax(out))
                pred_class = model.names[pred_idx]
            else:
                pred_class = model.names[out]
        else:
            pred_idx = int(np.argmax(out))
            pred_class = LABELS[pred_idx]

        pred_list.append(pred_class)

        # 5. Lưu sample 100 frames/lần
        if frame_idx % 100 == 0:
            cv2.imwrite(f"{save_dir}/samples/{true_label}_{frame_idx}.jpg", roi)

    cap.release()

    if len(pred_list) == 0:
        pred_list = ["nothing"]
        latencies = [0.0]

    return pred_list, latencies


# 6. Evaluate toàn bộ thư mục video

In [11]:
def evaluate_video_folder(video_dir, model_path, model_type, model_name):

    # Tạo thư mục lưu kết quả
    save_dir = f"/content/drive/MyDrive/AllModel/RealTime_Results/{model_name}"
    os.makedirs(save_dir, exist_ok=True)
    os.makedirs(f"{save_dir}/samples", exist_ok=True)

    # Load model classifier
    model = load_model_flexible(model_path, model_type)

    # Load YOLO hand detector
    hand_detector = YOLO("/content/drive/MyDrive/AllModel/HeThongPhienDichNNKH/model/hand_yolov8s.pt")  # sửa đúng file

    all_true = []
    all_pred = []
    all_latencies = []

    print("🚀 Bắt đầu test 29 video...")

    for fname in sorted(os.listdir(video_dir)):
        if not fname.lower().endswith((".mp4", ".avi", ".mov", ".mkv")):
            continue

        # Lấy nhãn từ tên file (khớp với LABELS)
        video_path = os.path.join(video_dir, fname)
        true_label = os.path.splitext(fname)[0]   # VD: "A.mp4" → "A"

        if true_label not in LABELS:
            print(f"⚠️ Bỏ qua file không hợp lệ: {fname}")
            continue

        print(f"▶ Testing video: {fname} (label={true_label})")

        pred_list, latencies = evaluate_single_video(
            video_path, true_label,
            model, model_type, hand_detector, save_dir
        )

        # flatten
        all_pred.extend(pred_list)
        all_true.extend([true_label]*len(pred_list))
        all_latencies.extend(latencies)

    # ---------- SAVE RESULTS ----------

    accuracy = np.mean(np.array(all_true)==np.array(all_pred))
    avg_latency = float(np.mean(all_latencies))
    avg_fps = 1000.0 / avg_latency

    # CSV
    df = pd.DataFrame({"true_label": all_true, "pred_label": all_pred})
    df.to_csv(f"{save_dir}/predictions.csv", index=False)

    # metrics
    report = classification_report(all_true, all_pred, labels=LABELS, output_dict=True)
    with open(f"{save_dir}/metrics.json","w") as f:
        json.dump({
            "model": model_name,
            "accuracy": accuracy,
            "report": report
        }, f, indent=4)

    # latency
    with open(f"{save_dir}/fps_latency.json","w") as f:
        json.dump({
            "avg_latency_ms": avg_latency,
            "avg_fps": avg_fps
        }, f, indent=4)

    # confusion matrix
    cm = confusion_matrix(all_true, all_pred, labels=LABELS)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=LABELS)
    fig, ax = plt.subplots(figsize=(12,12))
    disp.plot(ax=ax, xticks_rotation=45)
    plt.savefig(f"{save_dir}/confusion_matrix.png", dpi=200, bbox_inches='tight')
    plt.close()

    print("\n🎉 DONE!")
    print("📌 Accuracy:", accuracy)
    print("📌 FPS:", avg_fps)
    print("📁 Kết quả lưu tại:", save_dir)

In [12]:
VIDEO_DIR = "/content/drive/MyDrive/AllModel/Video"
MODEL_PATH = "/content/drive/MyDrive/AllModel/Model_New/rtdetr-l.pt"  # hoặc .pt
MODEL_TYPE = "yolo"                 #tf  hoặc "yolo" or vit
MODEL_NAME = "RTDETR1"

evaluate_video_folder(VIDEO_DIR, MODEL_PATH, MODEL_TYPE, MODEL_NAME)


Streaming output truncated to the last 5000 lines.

0: 384x640 1 hand, 366.2ms
Speed: 5.0ms preprocess, 366.2ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 hand, 369.5ms
Speed: 4.2ms preprocess, 369.5ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 hand, 371.6ms
Speed: 4.2ms preprocess, 371.6ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 hand, 369.2ms
Speed: 5.8ms preprocess, 369.2ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 hand, 392.3ms
Speed: 5.7ms preprocess, 392.3ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 hand, 396.1ms
Speed: 4.2ms preprocess, 396.1ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 hand, 407.8ms
Speed: 4.4ms preprocess, 407.8ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 hand, 385.4ms
Speed: 3.6ms preprocess, 385.4ms infere

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



🎉 DONE!
📌 Accuracy: 0.488212927756654
📌 FPS: 1.7302225736195882
📁 Kết quả lưu tại: /content/drive/MyDrive/AllModel/RealTime_Results/RTDETR1
